In [2]:
import torch
import numpy as np
import random, os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.set_device(0)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
def seed_it(seed):
    random.seed(seed)
    os.environ["PYTHONSEED"] = str(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.manual_seed(seed)
seed_it(42)

True


# NQ10

In [3]:
input_path = r'D:\data\lost-in-the-middle-main\qa_data\10_total_documents\nq-open-10_total_documents_gold_at_0.jsonl.gz'
# input_path = save_path

from typing import List, Optional, Tuple, Type, TypeVar
from copy import deepcopy
from pydantic.dataclasses import dataclass
import pathlib

In [4]:
from xopen import xopen
from tqdm import tqdm
import json, logging

from copy import deepcopy

logger = logging.getLogger()

examples = []
with xopen(input_path) as fin:
    for line in tqdm(fin):
        input_example = json.loads(line)

        examples.append(deepcopy(input_example))
logger.info(f"Loaded {len(examples)} prompts to process")

0it [00:00, ?it/s]

2655it [00:00, 5491.91it/s]


In [5]:
dataset_seed = 42
seed_it(dataset_seed)
all_index = list(range(len(examples)))
train_index = np.sort(random.sample(all_index, int(len(all_index)*0.8)))
test_index = np.sort(list(set(all_index).difference(set(train_index))))
print(f'prepare dataset, train size: {len(train_index)}, test size: {len(test_index)}')

prepare dataset, train size: 2124, test size: 531


In [6]:
import json
import pandas as pd
import math
from sentence_transformers import SentenceTransformer, util, InputExample, models
from transformers import AutoModelForMaskedLM, AutoTokenizer
import os
import numpy as np
from tqdm import tqdm
from datetime import datetime

In [7]:
# model_save_path = r'Models\nq_10_bert-2024-03-22_16-58-11'
# model_save_path = r'D:\model\contriever-msmarco'
# model_save_path = r'D:\model\contriever'
# model_save_path = r'D:\model\bge-reranker-large'
model_save_path = r'bert-base-uncased'

model = SentenceTransformer(model_save_path, )

No sentence-transformers model found with name D:\model\bert-base-uncased. Creating a new one with mean pooling.


In [8]:
from sentence_transformers.evaluation import RerankingEvaluator
def get_dual_dev(dataset, idx):
    dev_data = []
    for i in idx:
        data = dataset[i]
        query = data['question']
        pos = []
        neg = []
        for ctx in data['ctxs']:
            text = 'Title: '+ctx['title'] +'\n' + ctx['text']
            if ctx['isgold'] == 1:
                pos.append(text)
            else:
                neg.append(text)
        dev_data.append({'query': query, 'positive': pos, 'negative': neg})
    return dev_data
test_samples = get_dual_dev(examples, test_index)
dev_evaluator = RerankingEvaluator(test_samples, batch_size=64, show_progress_bar=True)

In [8]:
r = dev_evaluator(model, output_path='output/temp_result') # bert finetune 0.9601560398170568
r

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

{'map': 0.9601560398170568,
 'mrr@10': 0.9601560398170568,
 'ndcg@10': 0.9699027351592179}

In [10]:
r = dev_evaluator(model, output_path='output/temp_result') # contriever-ms 0.5681456072699012
r

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

{'map': 0.5681456072699012,
 'mrr@10': 0.5681456072699012,
 'ndcg@10': 0.6675529445365505}

In [16]:
r = dev_evaluator(model, output_path='output/temp_result') # contriever 0.5108742115804262
r

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

{'map': 0.5108742115804262,
 'mrr@10': 0.5108742115804262,
 'ndcg@10': 0.6251962795748911}

In [22]:
r = dev_evaluator(model, output_path='output/temp_result') # contriever bge 0.2226168056676531
r

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

{'map': 0.2226168056676531,
 'mrr@10': 0.2226168056676531,
 'ndcg@10': 0.3950224761658153}

In [9]:
r = dev_evaluator(model, output_path='output/temp_result') # contriever bert 0.3770424177203838
r

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

{'map': 0.3770424177203838,
 'mrr@10': 0.3770424177203838,
 'ndcg@10': 0.519053012879959}

In [10]:
from sentence_transformers.util import cos_sim
from tqdm import tqdm, trange
import numpy as np
from copy import deepcopy
import torch

def get_rank_list(q_emb, c_emb):
    cosine_score = cos_sim(q_emb, c_emb)[0].cpu()
    rank_list = np.array(cosine_score).argsort().tolist()[::-1]
    return rank_list

def get_dual_sim(dataset,idx, retrival_model):
    dataset_new = []
    for i in tqdm(idx):
        data = deepcopy(dataset[i])
        query = data['question']
        ctxs_text = []
        for ctx in data['ctxs']:
            text = 'Title: '+ctx['title'] +'\n' + ctx['text']
            ctxs_text.append(text)
        embs = retrival_model.encode([query] + ctxs_text, convert_to_tensor=True)
        q_emb = embs[0]
        c_emb = embs[1:]
        rank_list = get_rank_list(q_emb, c_emb)
        ctxs = []
        for i in range(len(rank_list)):
            j = rank_list[i]
            ctx = deepcopy(data['ctxs'][j])
            ctx['emb'] = c_emb[j].tolist()
            ctxs.append(ctx)
        data['ctxs'] = ctxs
        dataset_new.append(data)
    return dataset_new
res = get_dual_sim(examples, test_index[:10], model)

100%|██████████| 10/10 [00:00<00:00, 13.80it/s]


In [24]:
res = get_dual_sim(examples, all_index, model)

100%|██████████| 2655/2655 [08:34<00:00,  5.16it/s]


In [11]:
# save_path = 'dataset/nq-open-10_total_documents_gold_bert_emb.pkl'
# import pickle
# with open(save_path, 'wb') as fin:
#     pickle.dump(res, fin)
#     fin.close()

In [13]:
# save_path = 'dataset/nq-open-10_total_documents_gold_contriever_ms_emb.pkl'
# import pickle
# with open(save_path, 'wb') as fin:
#     pickle.dump(res, fin)
#     fin.close()

In [19]:
# save_path = 'dataset/nq-open-10_total_documents_gold_contriever_emb.pkl'
# import pickle
# with open(save_path, 'wb') as fin:
#     pickle.dump(res, fin)
#     fin.close()

In [25]:
# save_path = 'dataset/nq-open-10_total_documents_gold_bge_emb.pkl'
# import pickle
# with open(save_path, 'wb') as fin:
#     pickle.dump(res, fin)
#     fin.close()

In [11]:
res = get_dual_sim(examples, all_index, model)

100%|██████████| 2655/2655 [02:31<00:00, 17.48it/s]


In [31]:
# save_path = 'dataset/nq-open-10_total_documents_gold_bert_nft_emb.pkl'
# import pickle
# with open(save_path, 'wb') as fin:
#     pickle.dump(res, fin)
#     fin.close()

# HotpotQA

In [12]:
import torch
import numpy as np
import random, os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.set_device(0)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
def seed_it(seed):
    random.seed(seed)
    os.environ["PYTHONSEED"] = str(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.manual_seed(seed)
seed_it(42)

True


In [13]:
import json
with open(r'D:\data\hotpotqa\hotpot_train_v1.1.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)
    f.close()
with open(r'D:\data\hotpotqa\hotpot_dev_distractor_v1.json', 'r', encoding='utf-8') as f:
    dev_data = json.load(f)
    f.close()

In [14]:
count = 0
for data in train_data:
    if len(data['context']) == 10:
        count += len(data['supporting_facts'])
for data in dev_data:
    if len(data['context']) == 10:
        count += len(data['supporting_facts'])
count, count / (len(train_data) + len(dev_data))

(231742, 2.36829088828026)

In [15]:
print(f'prepare dataset, train size: {len(train_data)}, test size: {len(dev_data)}')

prepare dataset, train size: 90447, test size: 7405


In [16]:
import json
import pandas as pd
import math
from sentence_transformers import SentenceTransformer, util, InputExample, models
from transformers import AutoModelForMaskedLM, AutoTokenizer
import os
import numpy as np
from tqdm import tqdm
from datetime import datetime

In [17]:
model_name = r'hotpotqa_bert-2024-04-30_13-33-45'

In [18]:
max_seq_length = 512
word_embedding_model = models.Transformer(model_name, max_seq_length=max_seq_length)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

In [19]:
from sentence_transformers.evaluation import RerankingEvaluator
def get_dual_dev(dataset):
    dev_data = []
    for data in dataset:
        query = data['question']
        context = data['context']
        supporting_facts = data['supporting_facts']
        gt_titles = [s[0] for s in supporting_facts]
        pos = []
        neg = []
        for ctx in context:
            text = 'Title: '+ctx[0] +'\n' + ''.join(ctx[1])
            if ctx[0] in gt_titles:
                pos.append(text)
            else:
                neg.append(text)
        dev_data.append({'query': query, 'positive': pos, 'negative': neg})
    return dev_data
test_samples = get_dual_dev(dev_data)
dev_evaluator = RerankingEvaluator(test_samples, batch_size=64, show_progress_bar=True)

In [20]:
r = dev_evaluator(model, output_path='output/temp_result') # bert finetune 
r

Batches:   0%|          | 0/116 [00:00<?, ?it/s]

Batches:   0%|          | 0/1151 [00:00<?, ?it/s]

{'map': 0.917669585891417,
 'mrr@10': 0.9655916266831759,
 'ndcg@10': 0.9520248158480348}

In [21]:
from sentence_transformers.util import cos_sim
from tqdm import tqdm, trange
import numpy as np
from copy import deepcopy
import torch

def get_rank_list(q_emb, c_emb):
    cosine_score = cos_sim(q_emb, c_emb)[0].cpu()
    rank_list = np.array(cosine_score).argsort().tolist()[::-1]
    return rank_list

def get_dual_sim(dataset, retrival_model):
    dataset_new = []
    for data in tqdm(dataset):
        data = deepcopy(data)
        query = data['question']
        context = data['context']
        supporting_facts = data['supporting_facts']
        gt_titles = [s[0] for s in supporting_facts]
        ctxs_text = []
        for ctx in context:
            text = 'Title: '+ctx[0] +'\n' + ''.join(ctx[1])
            ctxs_text.append(text)
        embs = retrival_model.encode([query] + ctxs_text, convert_to_tensor=True)
        q_emb = embs[0]
        c_emb = embs[1:]
        rank_list = get_rank_list(q_emb, c_emb)
        ctxs = []
        for i in range(len(rank_list)):
            j = rank_list[i]
            ctx = deepcopy(data['context'][j])
            ctx.append(c_emb[j].tolist())
            ctxs.append(ctx)
        data['context'] = ctxs
        dataset_new.append(data)
    return dataset_new

In [22]:
res_t = get_dual_sim(dev_data[:1], model)
len(res_t[0]['context'][0][2])

100%|██████████| 1/1 [00:00<00:00,  4.85it/s]


768

In [23]:
res = get_dual_sim(train_data, model)

100%|██████████| 90447/90447 [2:12:39<00:00, 11.36it/s]  


In [24]:
res_test = get_dual_sim(dev_data, model)

100%|██████████| 7405/7405 [11:01<00:00, 11.20it/s]


In [27]:
save_path = 'dataset/hotpotqa/hotpot_train_v1.1_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res, fin)
    fin.close()

In [28]:
save_path = 'dataset/hotpotqa/hotpot_dev_distractor_v1_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res_test, fin)
    fin.close()

# MuSiQue

In [29]:
import torch
import numpy as np
import random, os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.set_device(0)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
def seed_it(seed):
    random.seed(seed)
    os.environ["PYTHONSEED"] = str(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.manual_seed(seed)
seed_it(42)

True


In [30]:
import json
with open(r'data\musique_v1.0\musique_ans_v1.0_train.jsonl', 'r', encoding='utf-8') as f:
    train_data = []
    for line in f.readlines():
        data = json.loads(line)
        if len(data['paragraphs']) != 20:
            continue
        train_data.append(data)
    f.close()
with open(r'data\musique_v1.0\musique_ans_v1.0_dev.jsonl', 'r', encoding='utf-8') as f:
    dev_data = []
    for line in f.readlines():
        data = json.loads(line)
        if len(data['paragraphs']) != 20:
            continue
        dev_data.append(data)
    f.close()

In [31]:
print(f'prepare dataset, train size: {len(train_data)}, test size: {len(dev_data)}')

prepare dataset, train size: 19917, test size: 2401


In [32]:
count = 0
for data in train_data:
    if len(data['paragraphs']) == 20:
        count += sum([1 if ctx['is_supporting'] else 0 for ctx in data['paragraphs']])
for data in dev_data:
    if len(data['paragraphs']) == 20:
        count += sum([1 if ctx['is_supporting'] else 0 for ctx in data['paragraphs']])
count, count / (len(train_data) + len(dev_data))
# (52929, 2.3676582420040257)
# (52929, 2.371583475221794)

(52929, 2.371583475221794)

In [33]:
import json
import pandas as pd
import math
from sentence_transformers import SentenceTransformer, util, InputExample, models
from transformers import AutoModelForMaskedLM, AutoTokenizer
import os
import numpy as np
from tqdm import tqdm
from datetime import datetime

In [34]:
model_save_path = r'Models\musique_bert-2024-04-30_19-45-08'

model = SentenceTransformer(model_save_path, )

In [35]:
from sentence_transformers.evaluation import RerankingEvaluator
def get_dual_dev(dataset):
    dev_data = []
    for data in dataset:
        query = data['question']
        paragraphs = data['paragraphs']
        pos = []
        neg = []
        for ctx in paragraphs:
            text = 'Title: '+ctx['title'] +'\n' + ctx['paragraph_text']
            if ctx['is_supporting']:
                pos.append(text)
            else:
                neg.append(text)
        dev_data.append({'query': query, 'positive': pos, 'negative': neg})
    return dev_data
test_samples = get_dual_dev(dev_data)
dev_evaluator = RerankingEvaluator(test_samples, batch_size=64, show_progress_bar=True)

In [36]:
r = dev_evaluator(model, output_path='output/temp_result') # bert finetune 
r

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   0%|          | 0/751 [00:00<?, ?it/s]

{'map': 0.572576924193833,
 'mrr@10': 0.7358379114522389,
 'ndcg@10': 0.6795441640932427}

In [37]:
from sentence_transformers.util import cos_sim
from tqdm import tqdm, trange
import numpy as np
from copy import deepcopy
import torch

def get_rank_list(q_emb, c_emb):
    cosine_score = cos_sim(q_emb, c_emb)[0].cpu()
    rank_list = np.array(cosine_score).argsort().tolist()[::-1]
    return rank_list

def get_dual_sim(dataset, retrival_model):
    dataset_new = []
    for data in tqdm(dataset):
        query = data['question']
        paragraphs = data['paragraphs']
        ctxs_text = []
        for ctx in paragraphs:
            text = 'Title: '+ctx['title'] +'\n' + ctx['paragraph_text']
            ctxs_text.append(text)
        embs = retrival_model.encode([query] + ctxs_text, convert_to_tensor=True)
        q_emb = embs[0]
        c_emb = embs[1:]
        rank_list = get_rank_list(q_emb, c_emb)
        ctxs = []
        for i in range(len(rank_list)):
            j = rank_list[i]
            ctx = deepcopy(data['paragraphs'][j])
            ctx['emb'] = c_emb[j].tolist()
            ctxs.append(ctx)
        data['paragraphs'] = ctxs
        dataset_new.append(data)
    return dataset_new

In [38]:
res_t = get_dual_sim(dev_data[:2], model)
[(r['emb']) for r in res_t[1]['paragraphs']]

100%|██████████| 2/2 [00:00<00:00,  6.30it/s]


[[0.5289779305458069,
  -0.3343210518360138,
  0.4443851113319397,
  1.121181607246399,
  0.07696312665939331,
  0.19747687876224518,
  0.11982018500566483,
  0.3240089416503906,
  0.008795341476798058,
  0.25058144330978394,
  -0.10157185047864914,
  0.1303057223558426,
  -0.0757875144481659,
  -0.09179003536701202,
  0.037231821566820145,
  0.12275470048189163,
  -0.21294745802879333,
  -0.4600921869277954,
  0.11305499076843262,
  -0.8674743175506592,
  -0.5065155625343323,
  -0.2597210705280304,
  0.055677756667137146,
  0.7850151658058167,
  0.011707122437655926,
  0.3930567502975464,
  0.381264328956604,
  0.07000071555376053,
  -0.9010576605796814,
  -0.17214271426200867,
  -0.02411416731774807,
  0.34694671630859375,
  0.0876360759139061,
  -0.04286053776741028,
  0.09265472739934921,
  0.5934671759605408,
  -0.5295015573501587,
  0.4850640892982483,
  -0.32819968461990356,
  -0.5259861946105957,
  -0.2055443376302719,
  0.06419198215007782,
  -0.37526315450668335,
  -0.5329461

In [39]:
res = get_dual_sim(train_data[:], model)

100%|██████████| 19917/19917 [58:27<00:00,  5.68it/s]    


In [40]:
res_test = get_dual_sim(dev_data[:], model)

100%|██████████| 2401/2401 [06:59<00:00,  5.72it/s]


In [41]:
save_path = 'dataset/musique/musique_ans_v1.0_train_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res, fin)
    fin.close()

In [42]:
save_path = 'dataset/musique/musique_ans_v1.0_dev_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res_test, fin)
    fin.close()

# 2Wiki

In [12]:
import torch
import numpy as np
import random, os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.set_device(0)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
def seed_it(seed):
    random.seed(seed)
    os.environ["PYTHONSEED"] = str(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.manual_seed(seed)
seed_it(42)

True


In [13]:
import json
with open(r'data\2wikimultihop\train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)
    f.close()
with open(r'data\2wikimultihop\dev.json', 'r', encoding='utf-8') as f:
    dev_data = json.load(f)
    f.close()

In [14]:
print(f'prepare dataset, train size: {len(train_data)}, test size: {len(dev_data)}')

prepare dataset, train size: 167454, test size: 12576


In [15]:
train_data = train_data[:8000]
dev_data = dev_data[:2000]

In [16]:
print(f'prepare dataset, train size: {len(train_data)}, test size: {len(dev_data)}')

prepare dataset, train size: 8000, test size: 2000


In [17]:
count = 0
for data in train_data:
    if len(data['context']) == 10:
        count += len(data['supporting_facts'])
for data in dev_data:
    if len(data['context']) == 10:
        count += len(data['supporting_facts'])
count, count / (len(train_data) + len(dev_data))

(24399, 2.4399)

In [18]:
import json
import pandas as pd
import math
from sentence_transformers import SentenceTransformer, util, InputExample, models
from transformers import AutoModelForMaskedLM, AutoTokenizer
import os
import numpy as np
from tqdm import tqdm
from datetime import datetime

In [19]:
model_save_path = r'Models\2wikimultihop_bert-2024-05-01_11-57-58'

model = SentenceTransformer(model_save_path, )

In [20]:
from sentence_transformers.evaluation import RerankingEvaluator
def get_dual_dev(dataset):
    dev_data = []
    for data in dataset:
        query = data['question']
        context = data['context']
        supporting_facts = data['supporting_facts']
        gt_titles = [s[0] for s in supporting_facts]
        pos = []
        neg = []
        for ctx in context:
            text = 'Title: '+ctx[0] +'\n' + ''.join(ctx[1])
            if ctx[0] in gt_titles:
                pos.append(text)
            else:
                neg.append(text)
        dev_data.append({'query': query, 'positive': pos, 'negative': neg})
    return dev_data
test_samples = get_dual_dev(dev_data)
dev_evaluator = RerankingEvaluator(test_samples, batch_size=64, show_progress_bar=True)

In [50]:
r = dev_evaluator(model, output_path='output/temp_result') # bert finetune 
r

Batches:   0%|          | 0/197 [00:00<?, ?it/s]

Batches:   0%|          | 0/1965 [00:00<?, ?it/s]

{'map': 0.9636680727210306,
 'mrr@10': 0.9907018659881255,
 'ndcg@10': 0.9803267635995991}

In [21]:
r = dev_evaluator(model, output_path='output/temp_result') # bert finetune 
r

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

{'map': 0.9613958333333333,
 'mrr@10': 0.9888083333333334,
 'ndcg@10': 0.9788876406326972}

In [22]:
from sentence_transformers.util import cos_sim
from tqdm import tqdm, trange
import numpy as np
from copy import deepcopy
import torch

def get_rank_list(q_emb, c_emb):
    cosine_score = cos_sim(q_emb, c_emb)[0].cpu()
    rank_list = np.array(cosine_score).argsort().tolist()[::-1]
    return rank_list

def get_dual_sim(dataset, retrival_model):
    dataset_new = []
    for data in tqdm(dataset):
        data = deepcopy(data)
        query = data['question']
        context = data['context']
        supporting_facts = data['supporting_facts']
        gt_titles = [s[0] for s in supporting_facts]
        ctxs_text = []
        for ctx in context:
            text = 'Title: '+ctx[0] +'\n' + ''.join(ctx[1])
            ctxs_text.append(text)
        embs = retrival_model.encode([query] + ctxs_text, convert_to_tensor=True)
        q_emb = embs[0]
        c_emb = embs[1:]
        rank_list = get_rank_list(q_emb, c_emb)
        ctxs = []
        for i in range(len(rank_list)):
            j = rank_list[i]
            ctx = deepcopy(data['context'][j])
            ctx.append(c_emb[j].tolist())
            ctxs.append(ctx)
        data['context'] = ctxs
        dataset_new.append(data)
    return dataset_new

In [23]:
res_t = get_dual_sim(dev_data[:2], model)
[r[2] for r in res_t[1]['context']]

100%|██████████| 2/2 [00:00<00:00,  7.63it/s]


[[0.18676485121250153,
  0.8349186182022095,
  -0.17394769191741943,
  -0.8274983167648315,
  0.07800401002168655,
  0.3407648503780365,
  0.8373925685882568,
  0.9545819759368896,
  -0.06016818806529045,
  -0.06165448948740959,
  -0.3397987186908722,
  -1.2180161476135254,
  -0.8719896078109741,
  0.6612454056739807,
  0.12004570662975311,
  -0.6565718650817871,
  0.4395337402820587,
  0.11975335329771042,
  -0.1348818689584732,
  0.8578444123268127,
  0.16354086995124817,
  -0.6419945359230042,
  -0.6250246167182922,
  -0.20298072695732117,
  -0.23549474775791168,
  -0.583430290222168,
  0.7128682136535645,
  -0.18069200217723846,
  -0.7949482798576355,
  -0.854210615158081,
  0.3875978887081146,
  -0.6939305663108826,
  -0.5592427849769592,
  0.32748717069625854,
  -0.4359569549560547,
  0.8580148220062256,
  -0.4771251678466797,
  0.34259551763534546,
  0.5764034390449524,
  0.4823397696018219,
  -0.7002056241035461,
  -0.22349223494529724,
  -0.5507374405860901,
  0.09660319983959

In [24]:
res = get_dual_sim(train_data, model)

100%|██████████| 8000/8000 [11:56<00:00, 11.17it/s]


In [25]:
res_test = get_dual_sim(dev_data[:], model)

100%|██████████| 2000/2000 [03:11<00:00, 10.44it/s]


In [26]:
save_path = 'dataset/2wikimultihop/2wikimultihop_train_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res, fin)
    fin.close()

In [27]:
save_path = 'dataset/2wikimultihop/2wikimultihop_dev_bert_emb.pkl'
import pickle

with open(save_path, 'wb') as fin:
    pickle.dump(res_test, fin)
    fin.close()